# CC200 → biological interpretation (UA-PoE, ESWA)

Fills **Table `tab:bio`** and the network-enrichment paragraph in §6.4.

## Why this notebook exists
CC200 is a **functional** parcellation: its parcels carry **indices, not names**. Turning "region 10" into *"left posterior cingulate"* requires the actual atlas volume — it cannot be done from the indices alone, and guessing would be fabrication. This notebook derives it deterministically from the atlas.

## What it produces
1. Centroid of each consensus parcel in **MNI152** space
2. Modal **AAL** / **Harvard–Oxford** label **+ overlap fraction**
3. Modal **Yeo-7 network** membership
4. **Hypergeometric enrichment** test per network (Holm-corrected)
5. Ready-to-paste **LaTeX** rows

> ### ⚠️ Verify the ROI order before trusting anything
> Your feature vector's column order must match this notebook's parcel numbering. If your pipeline reindexed, sorted, or dropped empty parcels, the indices shift silently and **the anatomy will be wrong**. Check `ROI_INDEX_BASE` in cell 2 against how you built your connectivity vectors.

In [1]:
# Run once. Needs internet (downloads ~100 MB of atlases on first run).
!pip install -q nilearn nibabel numpy scipy


[notice] A new release of pip is available: 23.1.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import nibabel as nib
from nilearn import datasets, image
from scipy.stats import hypergeom
print("imports ok")

imports ok


## 1. Configure

In [3]:
# The 10 parcels selected by all three attribution methods (Table 4 / Fig. fig:venn)
CONSENSUS = [10, 56, 62, 12, 113, 16, 70, 55, 88, 125]

# Are these ROI *labels* (1-based) or *column indices* (0-based) in your feature vector?
# ABIDE PCP rois_cc200 column j (0-based) = j-th ROI label in the header.
ROI_INDEX_BASE = 1     # 1 => CONSENSUS are ROI labels ; 0 => column indices

YEO_NAMES = {1: "Visual", 2: "Somatomotor", 3: "Dorsal attention",
             4: "Ventral attention / salience", 5: "Limbic",
             6: "Frontoparietal control", 7: "Default mode"}
print(f"{len(CONSENSUS)} consensus parcels, ROI_INDEX_BASE={ROI_INDEX_BASE}")

10 consensus parcels, ROI_INDEX_BASE=1


## 2. Fetch atlases

`scorr_mean` is 4-D (one map per clustering level). **Check that the index you pick is the k=200 map your pipeline used** — print the shape and confirm ~200 parcels.

In [4]:
cc = datasets.fetch_atlas_craddock_2012()
cc_img = image.load_img(image.index_img(cc["scorr_mean"], 0))   # <-- verify this index
cc_data = np.asarray(cc_img.dataobj)
present = sorted(int(v) for v in np.unique(cc_data) if v != 0)
print("CC200 map shape:", cc_img.shape, "| non-zero parcels:", len(present))
assert 150 < len(present) <= 220, "Not a ~200-parcel map — pick a different index_img()"

aal = datasets.fetch_atlas_aal()
ho  = datasets.fetch_atlas_harvard_oxford("cort-maxprob-thr25-2mm")
yeo = datasets.fetch_atlas_yeo_2011()

aal_r = image.resample_to_img(aal["maps"], cc_img, interpolation="nearest")
ho_r  = image.resample_to_img(ho["maps"],  cc_img, interpolation="nearest")
yeo_r = image.resample_to_img(image.index_img(yeo["thick_7"], 0), cc_img,
                              interpolation="nearest")
print("atlases resampled to CC200 grid")

[fetch_atlas_craddock_2012] Added README.md to C:\Users\AUC\nilearn_data
[fetch_atlas_craddock_2012] Dataset created in C:\Users\AUC\nilearn_data\craddock_2012
[fetch_atlas_craddock_2012] Downloading data from https://cluster_roi.projects.nitrc.org/Parcellations/craddock_2011_parcellations.tar.gz ...
[fetch_atlas_craddock_2012] Error while fetching file craddock_2011_parcellations.tar.gz; dataset fetching aborted.


SSLError: HTTPSConnectionPool(host='cluster_roi.projects.nitrc.org', port=443): Max retries exceeded with url: /Parcellations/craddock_2011_parcellations.tar.gz (Caused by SSLError(SSLCertVerificationError(1, "[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: Hostname mismatch, certificate is not valid for 'cluster_roi.projects.nitrc.org'. (_ssl.c:1002)")))

## 3. Map each consensus parcel

`Overlap` is the fraction of the parcel's voxels carrying the modal label. **Low overlap = the parcel straddles structures** and must be reported cautiously — this is why the paper's table has an overlap column.

In [ ]:
def modal_label(mask, atlas_img, labels, indices=None):
    a = np.asarray(atlas_img.dataobj)
    vals = a[mask]; vals = vals[vals != 0]
    if vals.size == 0:
        return "unlabelled", 0.0
    uniq, cnt = np.unique(vals, return_counts=True)
    best = int(uniq[np.argmax(cnt)]); frac = float(cnt.max() / mask.sum())
    if indices is not None:
        try:    name = labels[list(indices).index(best)]
        except (ValueError, IndexError): name = f"idx{best}"
    else:
        name = labels[best] if best < len(labels) else f"idx{best}"
    return (name.decode() if isinstance(name, bytes) else str(name)), frac

# network of EVERY parcel (needed for the enrichment null)
yeo_data = np.asarray(yeo_r.dataobj)
all_net = {}
for p in present:
    v = yeo_data[cc_data == p]; v = v[v != 0]
    all_net[p] = int(np.bincount(v.astype(int)).argmax()) if v.size else 0

rows = []
for rank, idx in enumerate(CONSENSUS, start=1):
    lab = idx if ROI_INDEX_BASE == 1 else present[idx]
    if lab not in present:
        print(f"!! parcel {lab} not in atlas — check ROI_INDEX_BASE"); continue
    m = cc_data == lab
    mni = nib.affines.apply_affine(cc_img.affine, np.argwhere(m).mean(0))
    aal_name, aal_f = modal_label(m, aal_r, aal["labels"], aal["indices"])
    ho_name,  ho_f  = modal_label(m, ho_r,  ho["labels"])
    nid = all_net[lab]; nname = YEO_NAMES.get(nid, "unassigned")
    v = yeo_data[m]; v = v[v != 0]
    net_f = float((v == nid).sum() / m.sum()) if v.size else 0.0
    rows.append(dict(cc=lab, aal=aal_name, aal_f=aal_f, ho=ho_name, ho_f=ho_f,
                     mni=mni, net=nname, net_f=net_f, rank=rank))
    print(f"CC {lab:3d} | {aal_name:26s} ({aal_f:4.0%}) | {nname:28s} ({net_f:4.0%}) "
          f"| MNI ({mni[0]:+4.0f},{mni[1]:+4.0f},{mni[2]:+4.0f})")

## 4. Network enrichment

Are the 10 parcels concentrated in particular networks, or spread as chance would predict? One-sided hypergeometric test per network, Holm-corrected across the 7.

**A null result is a real result** — a distributed, non-enriched map supports the paper's "network-level disorder, not a focal lesion" reading. Report it either way.

In [ ]:
N, n = len(present), len(rows)
raw = []
for nid, nname in YEO_NAMES.items():
    K = sum(1 for p in present if all_net[p] == nid)
    k = sum(1 for r in rows if r["net"] == nname)
    p = hypergeom.sf(k - 1, N, K, n) if k > 0 else 1.0
    raw.append((nname, k, K * n / N, p))

ps = np.array([r[3] for r in raw]); order = np.argsort(ps)
holm = np.empty_like(ps); prev = 0.0
for rank_, i in enumerate(order):
    prev = max(prev, min((len(ps) - rank_) * ps[i], 1.0)); holm[i] = prev

print(f"{'network':30s} {'obs':>4s} {'exp':>6s} {'p':>8s} {'Holm':>8s}")
print("-" * 60)
for (nname, k, exp, p), pc in zip(raw, holm):
    print(f"{nname:30s} {k:4d} {exp:6.1f} {p:8.3f} {pc:8.3f}{'  *' if pc < 0.05 else ''}")

## 5. LaTeX for `tab:bio`

In [ ]:
print("% --- replace the TODO rows of tab:bio with these ---")
for r in rows:
    x, y, z = r["mni"]
    lbl = r["aal"].replace("_", " ")
    print(f"{r['cc']} & {lbl} & $({x:+.0f},{y:+.0f},{z:+.0f})$ & "
          f"{r['aal_f']*100:.0f}\\% & {r['net']} & {r['rank']} \\\\")

## 6. Then what

1. Paste the rows into `tab:bio`.
2. Write the **network-level reading** paragraph from the cell-4 numbers (counts, expected, Holm-corrected *p*).
3. Write the **neurobiology** paragraph — **only for networks actually in your table**, citing primary sources, and framing agreement as *convergent validity for the attribution method*, not as a new neurobiological discovery.
4. Delete every `\TODO{}` from the paper.

⚠️ Any parcel with **low overlap (<~50%)** spans structures — say so in the text rather than presenting a clean label.

If you paste the outputs of cells 3–5 to me, I'll write both paragraphs in the paper's voice.